In [ ]:
import pandas as pd
import numpy as np
import torch

from src.forecaster import Forecaster
from src.models import LSTMHistoric, LSTMEncoderDecoder
from src.data.preprocessing import TimeSeriesDataPreprocessor
from src.utils.scores_and_losses import MAE, RMSE, DILATE

In [2]:
df = pd.read_csv('data/data.csv', index_col=1, parse_dates=True)

df

,Unnamed: 0,prcp,tmax,tmin,vp,srad,target
date,,,,,,,
1980-01-01,0,7.719036,2.635163,-7.404501,356.909575,108.665846,537.0
1980-01-02,1,3.660781,2.484138,-7.616933,350.118484,123.689874,570.0
1980-01-03,2,4.320509,0.016926,-5.137492,418.422967,74.293477,590.0
1980-01-04,3,13.075693,-1.130720,-15.443935,194.276939,143.881449,524.0
1980-01-05,4,15.739118,0.715120,-10.464246,281.074570,125.166372,594.0
...,...,...,...,...,...,...,...
2014-12-27,12770,14.423401,-3.103501,-6.586893,375.666898,71.825399,-999.0
2014-12-28,12771,10.665780,-2.270882,-8.514715,328.833729,112.132569,-999.0
2014-12-29,12772,0.000000,-8.595164,-19.983440,131.992812,202.927692,-999.0


In [3]:
df[df==-999.0] = np.nan
df_training = df[df.index < '2005-01-01']
df_validation = df[(df.index >= '2005-01-01') & (df.index < '2010-01-01')]
df_test = df[df.index >= '2010-01-01']


In [ ]:
HISTORIC_COLS = ['prcp', 'tmax', 'tmin', 'vp', 'srad']
FUTURE_COLS   = ['prcp', 'tmax', 'tmin', 'vp', 'srad']
TARGET_COL    = 'target'
HORIZON       = 7
SEQ_LEN       = 30

preprocessor = TimeSeriesDataPreprocessor(
    historic_input_sequence_length=SEQ_LEN,
    forecasting_horizon=HORIZON,
    historic_cols=HISTORIC_COLS,
    future_cols=FUTURE_COLS,
    target_col=TARGET_COL,
)

ds = preprocessor.input_dfs_to_tensordataset([df_training])
print(f'x_h: {ds[0][0].shape}  x_f: {ds[0][1].shape}  y: {ds[0][2].shape}')

In [ ]:
fc = Forecaster(
    model=LSTMHistoric,
    model_config={
        'LSTM_hidden_size': 16,
        'LSTM_num_layers': 1,
        'dropout_rate': 0.0,
    },
    name='LSTMHistoric_test',
    dfs_training_sets=[df_training],
    dfs_validation_sets=[df_validation],
    historic_cols=HISTORIC_COLS,
    future_cols=FUTURE_COLS,
    forecasting_horizon=HORIZON,
    historic_input_sequence_length=SEQ_LEN,
    loss_function='mae',
    validation_score='rmse',
    minimize_validation_score=True,
    save_path='models',
    batch_size=1024,
    number_of_epochs=5,
    use_torch_compile=False,
    save_weights_every_n_epochs=1,
)

print(f'LSTMHistoric parameters: {fc.compute_number_of_parameters()}')

In [ ]:
fc.fit()

In [ ]:
fc2 = Forecaster(
    model=LSTMEncoderDecoder,
    model_config={
        'LSTM_hidden_size': 16,
        'LSTM_num_layers': 1,
        'dropout_rate': 0.0,
        'encoder_downscale_layer_size': 8,
    },
    name='LSTMEncoderDecoder_test',
    dfs_training_sets=[df_training],
    dfs_validation_sets=[df_validation],
    historic_cols=HISTORIC_COLS,
    future_cols=FUTURE_COLS,
    forecasting_horizon=HORIZON,
    historic_input_sequence_length=SEQ_LEN,
    loss_function='mae',
    validation_score='rmse',
    minimize_validation_score=True,
    save_path='models',
    batch_size=1024,
    number_of_epochs=5,
    use_torch_compile=False,
    save_weights_every_n_epochs=1,
)

print(f'LSTMEncoderDecoder parameters: {fc2.compute_number_of_parameters()}')

In [ ]:
fc2.fit()